<a href="https://colab.research.google.com/github/SaraAlinejad/vae_test_1/blob/main/Alinejad_Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 5 - Deadline 4/6/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Sara Alinejad

**AI usage statement:**
AI tools were used for code autocompletion and to assist with minor formatting details such as Unicode symbols (e.g., Å) and axis labels.

### Task 1 - 10 points

Here we consider a alanine dipeptide in vacuum that is simple model often used to test methods. It consists of one alanine (Ala) residues and capping group. It is conformational dynamics can be understood in terms of the backbone dihedral angles $\Phi$ and $\Psi$.

![Alanine Dipeptide](https://github.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Assignment-5/AlanineDipeptideVac-1.png?raw=true)

The free energy landscape/surface $F(\Phi,\Psi)$ (see figure above) as function of $\Phi$ and $\Psi$ is known as the [Ramachandran plot](https://en.wikipedia.org/wiki/Ramachandran_plot). Note that here we are considering the system in vacuum so the energy landscape is different from what one would expect in solvent.

This system has two states, $C_{7\mathrm{eq}}$ and $C_{7\mathrm{ax}}$, where the $C_{7\mathrm{eq}}$ state is more populated one (i.e., lower in free energy).

Note that free energy landscape/surface $F(\Phi,\Psi)$ is related to the probability density function $P(\Phi,\Psi)$ through the following equation:

$$
F(\Phi,\Psi) = -k_{\mathrm{B}} T \ln P(\Phi,\Psi) + C
$$

where $T$ is the temperature, $k_{\mathrm{B}}$ is the Boltzmann constant, so that $k_{\mathrm{B}} T$ is the thermal energy, and $C$ is an unimportant constant. We also have that

$$
P(\Phi,\Psi) \propto \exp[-\beta F(\Phi,\Psi) ]
$$

where $\beta = (k_{\mathrm{B}} T)^{-1}$ is the inverse of the thermal energy.

We will consider dataset from molecular dynamics simulations, in particular parallel tempering (PT) simulations, that should give correct equilibrium sampling according to the Boltzmann distribution.

The dataset we consider is at temperture of 576 K. It is from a 100 ns PT simulations where we save variables every 10 ps, so we have 10000 samples.

The dataset is composed of two files. The `AlanineDipeptide_T-576K_Dihedrals.data` file includes the dihedral angles, $\Phi$, $\Psi$, $\Theta$. The `AlanineDipeptide_T-576K_HeavyAtomDistances.data` file includes all possible pairs of heavy atom distances (there are 10 heavy atoms, so the number of all possible pairs is $(10\cdot 9)/2 = 45$).

#### A)
Visualize the dataset in terms of the backbone dihedral angles $\Phi$ and $\Psi$. You should look at the timeseries and a 2D scatter plots.

You should estimate and visualize the 1D probability density for $\Phi$ and $\Psi$, and also the 2D probability density for $\Phi$ and $\Psi$.

Optional: You will get 1 extra point if you correctly handle the periodicity of the dihedral angles when estimating the 1D and 2D probability densities.

#### B)
Design criteria so that you can assign each sample from the dataset to either the $C_{7\mathrm{eq}}$ or the $C_{7\mathrm{ax}}$ state. You should use this later on to check if the embedding that you obtain correctly seperate the two differnet states.

Note: This criteria should be more advanced than what we did in Lecture 16 where we only considered if $\Phi$ was postive.

#### C)
Perform Principal Component Analysis (PCA) for the system using all heavy atom distances as input features, and visualize the first two principle components.

Here, you should use the full dataset in the analysis (all 10000 samples).

How much of the variance is described by the first two principle components?

Use the results from B) to understand if the PCA embeddings is able to seperate the $C_{7\mathrm{eq}}$ and $C_{7\mathrm{ax}}$ states.

You should try both using unscaled and scaled distances as input features. How does this affect the results.

#### D)
Some of the heavy atom distances might not be relevant and only add noise. To address this, design criteria for filtering out distances that might not be relevant and re-do the PCA analysis. How does this affect the results?

For example, you might consider only using distances that have standard deviation above some given threshold value.

#### E)
Perform *t*-SNE analysis using the [openTSNE](https://opentsne.readthedocs.io/en/stable/index.html) code and obtain two-dimensional embeddings.

One feature of the openTNSE implementation of *t*-SNE, as compared to the scikit-learn one, is that it allows for embedding new samples into existing embeddings.

Here, you should use 10% of the dataset (1000 samples) to fit the $t$-SNE embedding (for example by taking only every 10 sample), and then transform the full dataset and plot and visulize that.

Here, you should use the set of filtered distances that you used in D). I would also recommend to use scaled distances.

Here, you should consider different perplexity values in the analysis. What is optimal perplexity value in your opinion?

Once you have found the optimal perplexity value, you can you try to use the full dataset in the fitting of the $t$-SNE embedding. The openTNSE implementation should more performant thant the scikit-learn one. Does this affect the results in some manner.

#### F) - Optional for 2 point
For the *t*-SNE analysis, using the the optimal perplexity value, test the effect of 1) using either unscaled or scaled features; and 2) using either the filtered distances or the full set of 45 distances.

In [ ]:
# Download datasets

%%bash
datasets="
AlanineDipeptide_T-576K_Dihedrals.data
AlanineDipeptide_T-576K_HeavyAtomDistances.data
"

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-5/Datasets"

for d in ${datasets}
do
  wget ${url}/${d} &> /dev/null
done

ls

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
!pip install openTSNE -q
from openTSNE import TSNE

In [ ]:
!head AlanineDipeptide_T-576K_Dihedrals.data

In [ ]:
def get_variables_names_from_header(filename):
    with open(filename, 'r') as f:
        header = f.readline()
    return header.split()[2:]

dih_variables = get_variables_names_from_header("AlanineDipeptide_T-576K_Dihedrals.data")
data_dih = pd.read_csv("AlanineDipeptide_T-576K_Dihedrals.data",
                        header=None, names=dih_variables, sep=r"\s+", comment="#")

phi = data_dih["phi"].values
psi = data_dih["psi"].values
time = data_dih["time"].values

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(time, np.degrees(phi), '.', ms=1, color='blue', alpha=0.5)
axes[0].set_ylabel(r"$\Phi$ (deg)")
axes[0].set_ylim(-180, 180)
axes[0].set_yticks([-180, -90, 0, 90, 180])

axes[1].plot(time, np.degrees(psi), '.', ms=1, color='red', alpha=0.5)
axes[1].set_ylabel(r"$\Psi$ (deg)")
axes[1].set_xlabel("Time (ps)")
axes[1].set_ylim(-180, 180)
axes[1].set_yticks([-180, -90, 0, 90, 180])

plt.suptitle(r"Timeseries of $\Phi$ and $\Psi$")
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(np.degrees(phi), np.degrees(psi), s=1, alpha=0.3, color='blue')
ax.set_xlabel(r"$\Phi$ (deg)")
ax.set_ylabel(r"$\Psi$ (deg)")
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xticks([-180, -90, 0, 90, 180])
ax.set_yticks([-180, -90, 0, 90, 180])
ax.set_title(r"2D Scatter: $\Phi$ vs $\Psi$")
plt.tight_layout()
plt.show()


def periodic_kde_1d(angles_rad, n_points=500):

    x_eval = np.linspace(-np.pi, np.pi, n_points)
    tiled = np.concatenate([angles_rad - 2*np.pi, angles_rad, angles_rad + 2*np.pi])
    kde = gaussian_kde(tiled, bw_method=0.15)
    density = kde(x_eval) * 3   # 3 copies : rescale back
    return x_eval, density


def periodic_kde_2d(phi_rad, psi_rad, n_grid=200, sigma_deg=5):
    grid = np.linspace(-np.pi, np.pi, n_grid)
    dx = grid[1] - grid[0]

    counts, _, _ = np.histogram2d(phi_rad, psi_rad,
                                   bins=n_grid,
                                   range=[[-np.pi, np.pi], [-np.pi, np.pi]])
    sigma_bins = (np.radians(sigma_deg)) / dx
    density = gaussian_filter(counts, sigma=sigma_bins, mode='wrap')
    density = density / (density.sum() * dx**2)

    xx, yy = np.meshgrid(grid, grid, indexing='ij')
    return xx, yy, density


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_phi, d_phi = periodic_kde_1d(phi)
axes[0].plot(np.degrees(x_phi), d_phi, color='blue', lw=2, label=r"$P(\Phi)$")
axes[0].set_xlabel(r"$\Phi$ (deg)")
axes[0].set_ylabel("Probability density")
axes[0].set_xlim(-180, 180)
axes[0].set_xticks([-180, -90, 0, 90, 180])
axes[0].legend()
axes[0].set_title(r" $\Phi$ ")

x_psi, d_psi = periodic_kde_1d(psi)
axes[1].plot(np.degrees(x_psi), d_psi, color='red', lw=2, label=r"$P(\Psi)$")
axes[1].set_xlabel(r"$\Psi$ (deg)")
axes[1].set_ylabel("Probability density")
axes[1].set_xlim(-180, 180)
axes[1].set_xticks([-180, -90, 0, 90, 180])
axes[1].legend()
axes[1].set_title(r"$\Psi$ ")

plt.tight_layout()
plt.show()


xx, yy, density = periodic_kde_2d(phi, psi)

fig, ax = plt.subplots(figsize=(7, 6))
cf = ax.contourf(np.degrees(xx), np.degrees(yy), density,
                  levels=30, cmap="seismic")
plt.colorbar(cf, ax=ax, label=r"$P(\Phi,\Psi)$")
ax.set_xlabel(r"$\Phi$ (deg)")
ax.set_ylabel(r"$\Psi$ (deg)")
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xticks([-180, -90, 0, 90, 180])
ax.set_yticks([-180, -90, 0, 90, 180])
ax.set_title(r" $\Phi$–$\Psi$ Ramachandran")
plt.tight_layout()
plt.show()

In [ ]:
def assign_states(phi_rad, psi_rad):
    phi_deg = np.degrees(phi_rad)
    psi_deg = np.degrees(psi_rad)

    c7eq = (phi_deg < 0)
    c7ax = (phi_deg >= 0) & (phi_deg <= 120)

    labels = np.full(len(phi_rad), "unassigned", dtype=object)
    labels[c7eq] = "C7eq"
    labels[c7ax] = "C7ax"
    return labels

labels = assign_states(phi, psi)

print(f"C7eq  : {np.sum(labels == 'C7eq'):5d}  ({100*np.mean(labels == 'C7eq'):.1f}%)")
print(f"C7ax  : {np.sum(labels == 'C7ax'):5d}  ({100*np.mean(labels == 'C7ax'):.1f}%)")
print(f"Unassigned: {np.sum(labels == 'unassigned'):5d}  ({100*np.mean(labels == 'unassigned'):.1f}%)")

colors = {"C7eq": "blue", "C7ax": "red"}

fig, ax = plt.subplots(figsize=(6, 6))
for state, color in colors.items():
    mask = labels == state
    ax.scatter(np.degrees(phi[mask]), np.degrees(psi[mask]),
               s=1, alpha=0.4, color=color, label=state)

ax.set_xlabel(r"$\Phi$ (deg)")
ax.set_ylabel(r"$\Psi$ (deg)")
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xticks([-180, -90, 0, 90, 180])
ax.set_yticks([-180, -90, 0, 90, 180])
ax.legend(markerscale=6)
ax.set_title("State assignment on Ramachandran plot")
plt.tight_layout()
plt.show()

In [ ]:
dist_variables = get_variables_names_from_header("AlanineDipeptide_T-576K_HeavyAtomDistances.data")
data_dist = pd.read_csv("AlanineDipeptide_T-576K_HeavyAtomDistances.data",
                         header=None, names=dist_variables, sep=r"\s+", comment="#")
data_dist.reset_index(drop=True, inplace=True)
X = data_dist[dist_variables[1:]].values

color_map = {"C7eq": "blue", "C7ax": "red"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, scale, title in zip(axes, [False, True], ["Unscaled", "Scaled"]):
    X_input = StandardScaler().fit_transform(X) if scale else X

    pca = PCA(n_components=10)
    Z = pca.fit_transform(X_input)
    var1, var2 = pca.explained_variance_ratio_[:2] * 100

    for state, color in color_map.items():
        mask = np.array(labels) == state
        ax.scatter(Z[mask, 0], Z[mask, 1], s=1, alpha=0.4,
                   color=color, label=state)

    ax.set_xlabel(f"PC1 ({var1:.1f}%)")
    ax.set_ylabel(f"PC2 ({var2:.1f}%)")
    ax.set_title(f"PCA — {title} distances")
    ax.legend(markerscale=6)

plt.tight_layout()
plt.show()

# variance explained
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scale, title in zip(axes, [False, True], ["Unscaled", "Scaled"]):
    X_input = StandardScaler().fit_transform(X) if scale else X
    pca = PCA(n_components=10).fit(X_input)
    cumvar = np.cumsum(pca.explained_variance_ratio_) * 100

    ax.bar(range(1, 11), pca.explained_variance_ratio_ * 100,
           color="blue", alpha=0.7, label="Individual")
    ax.plot(range(1, 11), cumvar, 'o-', color="red", label="Cumulative")
    ax.set_xlabel("Principal component")
    ax.set_ylabel("Explained variance (%)")
    ax.set_title(f"Variance explained — {title}")
    ax.set_xticks(range(1, 11))
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# compute std of each distance (in Angstrom )
stds = X.std(axis=0)
dist_names = np.array(dist_variables[1:])

# plot std distribution to pick threshold
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.sort(stds) * 10, 'o-', color='blue', ms=4)
ax.axhline(y=0.1, color='red', ls='--', label='threshold = 0.1 Å')
ax.set_xlabel("Distance index (sorted by std)")
ax.set_ylabel("Std deviation (Å)")
ax.set_title("Sorted std deviations of heavy atom distances")
ax.legend()
plt.tight_layout()
plt.show()

# filter
threshold = 0.1 / 10
mask_feat = stds > threshold
X_filtered = X[:, mask_feat]
filtered_names = dist_names[mask_feat]

print(f"Kept {mask_feat.sum()} / {len(stds)} distances (threshold std > {threshold*10:.2f} Å)")

# PCA on filtered + scaled
color_map = {"C7eq": "blue", "C7ax": "red"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, scale, title in zip(axes, [False, True], ["Unscaled", "Scaled (filtered)"]):
    X_input = StandardScaler().fit_transform(X_filtered) if scale else X_filtered

    pca = PCA(n_components=10)
    Z = pca.fit_transform(X_input)
    var1, var2 = pca.explained_variance_ratio_[:2] * 100

    for state, color in color_map.items():
        m = np.array(labels) == state
        ax.scatter(Z[m, 0], Z[m, 1], s=1, alpha=0.4, color=color, label=state)

    ax.set_xlabel(f"PC1 ({var1:.1f}%)")
    ax.set_ylabel(f"PC2 ({var2:.1f}%)")
    ax.set_title(f"PCA filtered — {title}")
    ax.legend(markerscale=6)

plt.tight_layout()
plt.show()

# variance explained
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scale, title in zip(axes, [False, True], ["Unscaled", "Scaled"]):
    X_input = StandardScaler().fit_transform(X_filtered) if scale else X_filtered
    pca = PCA(n_components=10).fit(X_input)
    cumvar = np.cumsum(pca.explained_variance_ratio_) * 100

    ax.bar(range(1, 11), pca.explained_variance_ratio_ * 100,
           color="blue", alpha=0.7, label="Individual")
    ax.plot(range(1, 11), cumvar, 'o-', color="red", label="Cumulative")
    ax.set_xlabel("Principal component")
    ax.set_ylabel("Explained variance (%)")
    ax.set_title(f"Variance explained filtered — {title}")
    ax.set_xticks(range(1, 11))
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=10).fit(X_filtered)  # unscaled
Z = pca.transform(X_filtered)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
pairs = [(0,1),(0,2),(1,2),(0,3),(1,3),(2,3)]

for ax, (i,j) in zip(axes.ravel(), pairs):
    vi, vj = pca.explained_variance_ratio_[[i,j]] * 100
    for state, color in color_map.items():
        m = np.array(labels) == state
        ax.scatter(Z[m,i], Z[m,j], s=1, alpha=0.3, color=color, label=state)
    ax.set_xlabel(f"PC{i+1} ({vi:.1f}%)")
    ax.set_ylabel(f"PC{j+1} ({vj:.1f}%)")
    ax.legend(markerscale=6)

plt.tight_layout()
plt.show()

PC4 captures a much smaller but more conformationally specific variance that happens to be the geometric difference between C7ax and C7eq. let's look at which distances load most heavily on PC4:


In [ ]:
pca = PCA(n_components=4)
pca.fit(StandardScaler().fit_transform(X_filtered))

loadings_pc4 = pd.Series(pca.components_[3], index=filtered_names)
loadings_pc4.abs().sort_values(ascending=False).head(10).plot(
    kind='bar', color='deepskyblue', figsize=(10, 4)
)
plt.ylabel("Absolute loading on PC4")
plt.title("Top contributing distances to PC4")
plt.tight_layout()
plt.show()

In [ ]:
# filtered + scaled distances from D
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# 10% subset for fitting
X_fit = X_scaled[::10]
labels_fit = np.array(labels)[::10]

color_map = {"C7eq": "mediumblue", "C7ax": "crimson"}

perplexities = [10, 30, 50, 100, 200]

fig, axes = plt.subplots(1, len(perplexities), figsize=(5*len(perplexities), 5))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(perplexity=perp, n_components=2, random_state=42,
                n_jobs=-1, verbose=False)
    embedding_fit = tsne.fit(X_fit)
    embedding_full = embedding_fit.transform(X_scaled)

    for state, color in color_map.items():
        m = np.array(labels) == state
        ax.scatter(embedding_full[m, 0], embedding_full[m, 1],
                   s=1, alpha=0.3, color=color, label=state)

    ax.set_title(f"perplexity = {perp}")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(markerscale=6)

plt.suptitle("t-SNE embeddings — fit on 10%, transform full dataset", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
best_perp = 30

configs = {
    "Scaled + Filtered":   (X_filtered, True),
    "Unscaled + Filtered": (X_filtered, False),
    "Scaled + Full":       (X,          True),
    "Unscaled + Full":     (X,          False),
}

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for ax, (title, (X_base, scale)) in zip(axes, configs.items()):
    X_input = StandardScaler().fit_transform(X_base) if scale else X_base

    X_input_fit = X_input[::10]

    tsne = TSNE(perplexity=best_perp, n_components=2, random_state=42,
                n_jobs=-1, verbose=False)
    emb = tsne.fit(X_input_fit).transform(X_input)

    for state, color in color_map.items():
        m = np.array(labels) == state
        ax.scatter(emb[m, 0], emb[m, 1], s=1, alpha=0.3, color=color, label=state)

    ax.set_title(title)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(markerscale=6)

plt.suptitle(f"t-SNE comparison — perplexity = {best_perp}", y=1.02)
plt.tight_layout()
plt.show()